In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

In [4]:
ROOT = next(
    (    
        path for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "pyproject.toml").exists()
    ),
    None
)

if ROOT is None:
    raise FileNotFoundError("Project root not found")

RAW_DIR = ROOT / "data" / "raw"
REPORT_DIR = ROOT / "reports"

REPORT_DIR.mkdir(exist_ok=True)

# Expected input files
expected_files = {
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv",
}

csv_files = sorted(RAW_DIR.glob("*.csv"))
found_files = {file.name for file in csv_files}

missing = expected_files - found_files

if missing:
    raise FileNotFoundError(
        f"Missing dataset files: {sorted(missing)}"
    )

print("Dataset verified successfully!")
print(f"Total CSV files: {len(csv_files)}")

for file in csv_files:
    print(file.name)


Dataset verified successfully!
Total CSV files: 9
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [ ]:
table_profiles = []
column_profiles = []

for file in csv_files:
    df = pd.read_csv(file, low_memory=False)

    table_profiles.append({
        "table": file.stem,
        "rows": len(df),
        "columns": df.shape[1],
        "duplicate_rows": int(df.duplicated().sum()),
        "total_missing": int(df.isna().sum().sum())
    })

    for col in df.columns:
        column_profiles.append({
            "table": file.stem,
            "column": col,
            "data_type": str(df[col].dtype),
            "missing_values": int(df[col].isna().sum()),
            "missing_pct": round(
                df[col].isna().mean() * 100, 2
            ),
            "unique_values": int(df[col].nunique())
        })

    del df

table_profile = pd.DataFrame(table_profiles)
column_profile = pd.DataFrame(column_profiles)

print("=== TABLE OVERVIEW ===")
display(table_profile)

print("=== COLUMNS WITH MISSING VALUES ===")
display(
    column_profile[
        column_profile["missing_values"] > 0
    ].sort_values(
        "missing_pct", ascending=False
    ).head(20)
)

# Export reports
table_profile.to_csv(
    REPORT_DIR / "table_profile.csv",
    index=False
)

column_profile.to_csv(
    REPORT_DIR / "column_profile.csv",
    index=False
)

print("Data quality reports exported.")
